# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a FAIR² Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Make sure the `mlcroissant` library is installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We will inspect the dataset metadata and get an overview of what it contains.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}\n")
print(f"Authors: \n- " + "\n- ".join([str(a['@id']) for a in getattr(metadata, 'author', [])]))


## 2. Data Overview
Let's examine what record sets are available and which fields (columns) are included.

All entities (record sets, fields, etc.) are referenced by their `@id` as per the Croissant schema.

In [ ]:
# Helper: Get all RecordSet IDs and their fields
record_set_ids = []

print("Available Record Sets in this dataset:")
for rs in metadata.record_sets:
    rid = rs['@id']
    record_set_ids.append(rid)
    print(f"- RecordSet `@id`: {rid}")
    field_ids = [f['@id'] for f in rs.get('field', [])]
    print(f"  Fields: {field_ids}\n")

# For demonstration, print fields for first record set
if record_set_ids:
    print(f"Example fields for `{record_set_ids[0]}`:")
    for f in metadata.record_sets[0].get('field', []):
        print(f"- Field `@id`: {f['@id']} (dataType: {f.get('dataType', 'unknown')})")

## 3. Data Extraction
We now extract data from each record set into pandas DataFrames. All extraction operations reference record sets and fields by their `@id` as per the FAIR² data standard.

In [ ]:
# Load records from each record set into a DataFrame
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {df.shape[0]} records for Record Set '@id': {rs_id}, columns: {list(df.columns)}\n")
        else:
            print(f"[Warning] No records found for Record Set '@id': {rs_id}")
    except Exception as e:
        print(f"[Error] Could not load records for Record Set '@id': {rs_id} -- {e}")

# For illustration, show the head of the first record set loaded
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"DataFrame columns for Record Set '@id': {main_rs_id}")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print('No dataframes were loaded - check the dataset definition.')

## 4. Exploratory Data Analysis (EDA)
We perform basic analysis using fields as referenced by their `@id`. We'll demonstrate filtering based on a numeric field, normalization, and grouping.

**Note**: Replace `<numeric_field_id>` and `<group_field_id>` with actual IDs from the overview section, adjust thresholds as appropriate for your dataset.

In [ ]:
# Pick example field ids for demo (let's check for a GAD-7 or PHQ-9 score field)
# You may need to adapt this if field IDs differ.
example_numeric_field = None
example_group_field = None
main_df = None

if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    main_df = dataframes[main_rs_id]
    # List columns
    print("Available columns (field @id) in main RecordSet:")
    print(main_df.columns.tolist())
    # Try to find 'gad7_total_score' or 'phq9_total_score' or similar
    for c in main_df.columns:
        if 'gad' in c.lower() and 'total' in c.lower():
            example_numeric_field = c
            break
    if not example_numeric_field:
        for c in main_df.columns:
            if 'phq' in c.lower() and 'total' in c.lower():
                example_numeric_field = c
                break
    # As group field, pick e.g. 'village' or 'level_of_education' if available
    for c in main_df.columns:
        if 'village' in c.lower():
            example_group_field = c
            break
    if not example_group_field:
        for c in main_df.columns:
            if 'education' in c.lower():
                example_group_field = c
                break

if main_df is not None and example_numeric_field is not None:
    print(f"Using numeric_field = '{example_numeric_field}'")

    # Convert to numeric (in case of string)
    main_df[example_numeric_field] = pd.to_numeric(main_df[example_numeric_field], errors='coerce')

    # Choose a threshold (demonstration: scores > 10)
    threshold = 10
    filtered_df = main_df[main_df[example_numeric_field] > threshold].copy()
    print(f"\nNumber of records with {example_numeric_field} > {threshold}: {len(filtered_df)}\n")
    display(filtered_df[[example_numeric_field]].head(10))

    # Normalize the numeric field
    filtered_df[f"{example_numeric_field}_normalized"] = (
        (filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()) /
        filtered_df[example_numeric_field].std()
    )
    print(f"\nNormalized '{example_numeric_field}':\n")
    display(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head(10))

    # Group by a categorical field (if available)
    if example_group_field and example_group_field in main_df.columns:
        print(f"\nGrouping by '{example_group_field}':\n")
        grouped = filtered_df.groupby(example_group_field)[example_numeric_field].mean().sort_values(ascending=False)
        print(grouped.head(10))
else:
    print("No numeric field available for EDA. Please check data extraction result.")

## 5. Visualization
We visualize the distribution or relationships between fields using matplotlib/seaborn. (Feel free to adapt the visual below to your analysis needs.)

In [ ]:
# Visualize the distribution of the chosen numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and example_numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[example_numeric_field].dropna(), bins=15, kde=True, color='skyblue')
    plt.title(f"Distribution of {example_numeric_field}")
    plt.xlabel(example_numeric_field)
    plt.ylabel('Count')
    plt.show()

    if example_group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=example_group_field, y=example_numeric_field, data=main_df)
        plt.title(f"{example_numeric_field} by {example_group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Visualization skipped: required field(s) not found.')

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using Croissant metadata and the `mlcroissant` Python library.

- The dataset was accessed and pre-explored using record and field `@id`s for traceable, FAIR-aligned analysis.
- Basic filtering, normalization, and visualizations were applied to a survey-derived numeric field (e.g., GAD-7 or PHQ-9 score), and records were grouped by a demographic attribute (e.g., village or education level).

You can extend this notebook to perform deeper statistical analysis, feature engineering, or machine learning with the extracted, AI-ready data!

---
🌍 *For additional details about this dataset, refer to the Croissant metadata or [dataset DOI link](https://sen.science/doi/10.71728/senscience.vcs2-05nj).*